# 05. Quantum Annealing 실행

## 절차

```
QUBO 생성 -> 변수 수 확인 -> QPU embedding 시도
   embedding 가능?  YES -> QA 실행
                    NO  -> NOT_EMBEDDABLE 기록
```

## 고정 사항 (튜닝하지 않음)

- chain strength: `alpha * max|J|` 규칙으로 고정. instance마다 QUBO 계수 스케일이 크게 다르므로, 절대값을 고정하면 오히려 instance 간 비교 조건이 불공정해진다.
- annealing time, num_reads: 설정 파일 고정값
- embedding: seed / timeout / tries 고정 → 판정이 재현 가능

## embedding 실패 시

QUBO를 축소하거나 변수를 제거하거나 formulation을 바꾸지 않는다. 그대로 `NOT_EMBEDDABLE`로 기록하고, 해당 instance의 Gurobi/SA 결과는 정상적으로 보고한다.

**실행 전 준비**: 환경변수 `DWAVE_API_TOKEN`을 설정하거나 `dwave config create`로 토큰을 등록해야 한다. 토큰이 없으면 이 notebook은 `NO_QPU_ACCESS`를 기록하고 정상 종료한다.

In [1]:
# 프로젝트 루트를 import 경로에 추가한다.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"
SOLUTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("설정 로드 완료:", len(config["instances"]), "개 instance")


설정 로드 완료: 4 개 instance


In [4]:
from src import qa_solver
from src.data_generator import CFLPInstance
from src.persistence import save_samples, save_table
from src.qubo_builder import build_qubo

margin = float(config["penalty"]["margin"])
precision = int(config["encoding"]["precision"])
feas_tolerance = float(config["feasibility"]["tolerance"])

instances = [
    CFLPInstance.load(DATA_DIR / f"{spec['name']}.json")
    for spec in config["instances"]
]

LINKING_VARIANTS = (False, True)

records = []
best_samples = {}
for instance in instances:
    for formulation in ("SS", "MS"):
      for include_linking in LINKING_VARIANTS:
        model = build_qubo(
            instance, formulation, margin, precision,
            include_linking=include_linking,
        )
        outcome, embedding = qa_solver.solve(
            model,
            instance,
            config["qa"],
            config["embedding"],
            feas_tolerance,
        )
        record = {
            "instance": instance.name,
            "size": instance.num_customers,
            "formulation": formulation,
            "linking": include_linking,
            "qubo_variables": model.num_variables,
            "qubo_quadratic_terms": model.num_quadratic_terms,
        }
        record.update(outcome.to_record())
        records.append(record)
        tag = "L" if include_linking else "N"
        key = f"{instance.name}__{formulation}__{tag}"
        if outcome.best_sample is not None:
            best_samples[key + "__best"] = outcome.best_sample
        if outcome.best_feasible_sample is not None:
            best_samples[key + "__best_feasible"] = outcome.best_feasible_sample
        print(
            f"{instance.name} {formulation} linking={str(include_linking):5s} "
            f"vars={model.num_variables:5d}: status={outcome.status}"
        )
        if outcome.status != "OK":
            print("   ", outcome.extra.get("qa_message", ""))

qa_results = pd.DataFrame(records)
save_table(qa_results, RAW_DIR, "qa_results.csv")
if best_samples:
    save_samples(best_samples, RAW_DIR, "qa_best_samples.npz")
qa_results

4x4 SS linking=False vars=   44: status=OK
4x4 SS linking=True  vars=   60: status=OK
4x4 MS linking=False vars=  120: status=OK
4x4 MS linking=True  vars=  212: status=OK
6x6 SS linking=False vars=   79: status=OK
6x6 SS linking=True  vars=  115: status=OK
6x6 MS linking=False vars=  259: status=NOT_EMBEDDABLE
    QUBO 변수 259개를 QPU 그래프에 embedding하지 못했습니다 (seed=2024, timeout=300s).
6x6 MS linking=True  vars=  475: status=NOT_EMBEDDABLE
    QUBO 변수 475개를 QPU 그래프에 embedding하지 못했습니다 (seed=2024, timeout=300s).
8x8 SS linking=False vars=  117: status=OK
8x8 SS linking=True  vars=  181: status=OK
8x8 MS linking=False vars=  413: status=NOT_EMBEDDABLE
    QUBO 변수 413개를 QPU 그래프에 embedding하지 못했습니다 (seed=2024, timeout=300s).
8x8 MS linking=True  vars=  773: status=NOT_EMBEDDABLE
    QUBO 변수 773개를 QPU 그래프에 embedding하지 못했습니다 (seed=2024, timeout=300s).
15x15 SS linking=False vars=  329: status=OK
15x15 SS linking=True  vars=  554: status=NOT_EMBEDDABLE
    QUBO 변수 554개를 QPU 그래프에 embedding하지 못했습니다 (

,instance,size,formulation,linking,qubo_variables,qubo_quadratic_terms,solver,status,runtime,num_reads,best_energy,best_objective,best_is_feasible,best_total_violation,best_max_violation,feasible_fraction,best_feasible_objective,best_feasible_energy,num_samples,energy_mismatch,reported_best_energy,argmin_agreement,unique_samples,feasible_reads,qa_chain_strength,qa_chain_strength_alpha,qa_annealing_time,qa_chain_break_method,qa_chain_break_fraction,qa_wall_clock,qa_qpu_access_time_us,qa_qpu_sampling_time_us,embedding_status,logical_variables,physical_qubits,max_chain_length,mean_chain_length,embedding_search_time,qa_message
0,4x4,4,SS,False,44,246,QA,OK,1.239509,1000,4.516780e+04,3747.6264,False,3.0,1.0,0.003,3070.7826,1.222007e+06,1000,1.186677e-15,4.516780e+04,True,1000.0,3.0,1.149410e+07,1.5,20.0,majority_vote,0.018977,1.239509,218138.76,202380.0,OK,44,105,4,2.386364,0.627146,NaN
1,4x4,4,SS,True,60,278,QA,OK,1.215166,1000,2.389643e+05,2277.6464,False,1.0,1.0,0.003,3085.3246,3.038592e+06,1000,9.675025e-16,2.389643e+05,True,1000.0,3.0,1.149854e+07,1.5,20.0,majority_vote,0.004517,1.215166,175516.36,159760.0,OK,60,121,5,2.016667,0.694403,NaN
2,4x4,4,MS,False,120,2540,QA,OK,1.595993,1000,7.611137e+05,3716.2390,False,16.0,8.0,0.000,NaN,NaN,1000,3.892817e-15,7.611137e+05,True,1000.0,0.0,9.940841e+06,1.5,20.0,majority_vote,0.000508,1.595993,234596.76,218840.0,OK,120,1060,16,8.833333,29.264200,NaN
3,4x4,4,MS,True,212,3384,QA,OK,1.607709,1000,5.329211e+06,3760.8429,False,23.0,9.0,0.000,NaN,NaN,1000,8.734343e-15,5.329211e+06,True,1000.0,0.0,9.940841e+06,1.5,20.0,majority_vote,0.002354,1.607709,257379.96,241620.0,OK,212,1555,18,7.334906,46.004541,NaN
4,6x6,6,SS,False,79,571,QA,OK,1.189113,1000,3.054411e+06,6281.4159,False,11.0,6.0,0.000,NaN,NaN,1000,2.610729e-15,3.054411e+06,True,1000.0,0.0,3.356186e+07,1.5,20.0,majority_vote,0.009899,1.189113,142398.76,126640.0,OK,79,271,7,3.430380,2.448914,NaN
5,6x6,6,SS,True,115,643,QA,OK,1.221816,1000,2.976467e+06,6161.2962,False,3.0,1.0,0.000,NaN,NaN,1000,3.347319e-15,2.976467e+06,True,1000.0,0.0,3.357158e+07,1.5,20.0,majority_vote,0.029626,1.221816,161919.16,146160.0,OK,115,341,8,2.965217,2.050904,NaN
6,6x6,6,MS,False,259,8701,QA,NOT_EMBEDDABLE,NaN,0,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT_EMBEDDABLE,259,0,0,NaN,307.453460,QUBO 변수 259개를 QPU 그래프에 embedding하지 못했습니다 (seed...
7,6x6,6,MS,True,475,10753,QA,NOT_EMBEDDABLE,NaN,0,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT_EMBEDDABLE,475,0,0,NaN,307.825622,QUBO 변수 475개를 QPU 그래프에 embedding하지 못했습니다 (seed...
8,8x8,8,SS,False,117,1030,QA,OK,1.343865,1000,3.773204e+07,6583.5495,False,40.0,15.0,0.000,NaN,NaN,1000,2.645213e-15,3.773204e+07,True,1000.0,0.0,1.263163e+08,1.5,20.0,majority_vote,0.000530,1.343865,195956.36,180200.0,OK,117,619,11,5.290598,4.375632,NaN
9,8x8,8,SS,True,181,1158,QA,OK,1.356410,1000,3.982366e+07,7349.4479,False,73.0,20.0,0.000,NaN,NaN,1000,3.719087e-15,3.982366e+07,True,1000.0,0.0,1.263163e+08,1.5,20.0,majority_vote,0.000503,1.356410,213739.56,197980.0,OK,181,720,12,3.977901,7.641396,NaN


## embedding 결과 요약

instance 크기별 embedding 가능/불가능은 그 자체로 중요한 실험 결과이다.

In [5]:
columns = [
    column
    for column in (
        "instance",
        "formulation",
        "qubo_variables",
        "status",
        "embedding_status",
        "physical_qubits",
        "max_chain_length",
        "mean_chain_length",
        "embedding_search_time",
    )
    if column in qa_results.columns
]
qa_results[columns]

,instance,formulation,qubo_variables,status,embedding_status,physical_qubits,max_chain_length,mean_chain_length,embedding_search_time
0,4x4,SS,44,OK,OK,105,4,2.386364,0.627146
1,4x4,SS,60,OK,OK,121,5,2.016667,0.694403
2,4x4,MS,120,OK,OK,1060,16,8.833333,29.264200
3,4x4,MS,212,OK,OK,1555,18,7.334906,46.004541
4,6x6,SS,79,OK,OK,271,7,3.430380,2.448914
5,6x6,SS,115,OK,OK,341,8,2.965217,2.050904
6,6x6,MS,259,NOT_EMBEDDABLE,NOT_EMBEDDABLE,0,0,NaN,307.453460
7,6x6,MS,475,NOT_EMBEDDABLE,NOT_EMBEDDABLE,0,0,NaN,307.825622
8,8x8,SS,117,OK,OK,619,11,5.290598,4.375632
9,8x8,SS,181,OK,OK,720,12,3.977901,7.641396


## 실행에 성공한 경우의 지표

QPU access time과 wall-clock을 구분해 기록한다.

## linking constraint 비교

QA에서 가장 중요한 관찰 지점은 **embedding 성공 여부가 뒤집히는가**이다. linking constraint는 feasible region을 바꾸지 않으므로, embedding이 실패한다면 그것은 순수하게 QUBO 표현이 커진 대가이다.

In [6]:
from src.comparison import embedding_summary, formulation_gap, linking_effect

print("[1] linking 유무 (같은 formulation 내)")
display(linking_effect(qa_results))

[1] linking 유무 (같은 formulation 내)


,instance,size,formulation,vars_without,vars_with,vars_ratio,terms_without,terms_with,terms_ratio,feasible_without,feasible_with,objective_without,objective_with,runtime_without,runtime_with,status_without,status_with
1,4x4,4,MS,120,212,1.77,2540,3384,1.33,0.000,0.000,NaN,NaN,1.595993,1.607709,OK,OK
3,6x6,6,MS,259,475,1.83,8701,10753,1.24,NaN,NaN,NaN,NaN,NaN,NaN,NOT_EMBEDDABLE,NOT_EMBEDDABLE
5,8x8,8,MS,413,773,1.87,17603,20843,1.18,NaN,NaN,NaN,NaN,NaN,NaN,NOT_EMBEDDABLE,NOT_EMBEDDABLE
7,15x15,15,MS,1439,2774,1.93,123857,136427,1.10,NaN,NaN,NaN,NaN,NaN,NaN,NOT_EMBEDDABLE,NOT_EMBEDDABLE
0,4x4,4,SS,44,60,1.36,246,278,1.13,0.003,0.003,3070.7826,3085.3246,1.239509,1.215166,OK,OK
2,6x6,6,SS,79,115,1.46,571,643,1.13,0.000,0.000,NaN,NaN,1.189113,1.221816,OK,OK
4,8x8,8,SS,117,181,1.55,1030,1158,1.12,0.000,0.000,NaN,NaN,1.343865,1.356410,OK,OK
6,15x15,15,SS,329,554,1.68,5026,5476,1.09,0.000,NaN,NaN,NaN,2.011222,NaN,OK,NOT_EMBEDDABLE


In [7]:
print("[2] linking 포함끼리: SS vs MS")
display(formulation_gap(qa_results, linking=True))
print("[3] linking 제외끼리: SS vs MS")
display(formulation_gap(qa_results, linking=False))

[2] linking 포함끼리: SS vs MS


,instance,size,linking,vars_SS,vars_MS,vars_MS_over_SS,terms_SS,terms_MS,terms_MS_over_SS,feasible_SS,feasible_MS,objective_SS,objective_MS,status_SS,status_MS
0,4x4,4,True,60,212,3.53,278,3384,12.17,0.003,0.0,3085.3246,NaN,OK,OK
1,6x6,6,True,115,475,4.13,643,10753,16.72,0.000,NaN,NaN,NaN,OK,NOT_EMBEDDABLE
2,8x8,8,True,181,773,4.27,1158,20843,18.00,0.000,NaN,NaN,NaN,OK,NOT_EMBEDDABLE
3,15x15,15,True,554,2774,5.01,5476,136427,24.91,NaN,NaN,NaN,NaN,NOT_EMBEDDABLE,NOT_EMBEDDABLE


[3] linking 제외끼리: SS vs MS


,instance,size,linking,vars_SS,vars_MS,vars_MS_over_SS,terms_SS,terms_MS,terms_MS_over_SS,feasible_SS,feasible_MS,objective_SS,objective_MS,status_SS,status_MS
0,4x4,4,False,44,120,2.73,246,2540,10.33,0.003,0.0,3070.7826,NaN,OK,OK
1,6x6,6,False,79,259,3.28,571,8701,15.24,0.000,NaN,NaN,NaN,OK,NOT_EMBEDDABLE
2,8x8,8,False,117,413,3.53,1030,17603,17.09,0.000,NaN,NaN,NaN,OK,NOT_EMBEDDABLE
3,15x15,15,False,329,1439,4.37,5026,123857,24.64,0.000,NaN,NaN,NaN,OK,NOT_EMBEDDABLE


In [8]:
print("embedding 요약")
embedding_summary(qa_results)

embedding 요약


,instance,size,formulation,linking,qubo_variables,qubo_quadratic_terms,status,physical_qubits,max_chain_length
2,4x4,4,MS,False,120,2540,OK,1060,16
3,4x4,4,MS,True,212,3384,OK,1555,18
6,6x6,6,MS,False,259,8701,NOT_EMBEDDABLE,0,0
7,6x6,6,MS,True,475,10753,NOT_EMBEDDABLE,0,0
10,8x8,8,MS,False,413,17603,NOT_EMBEDDABLE,0,0
11,8x8,8,MS,True,773,20843,NOT_EMBEDDABLE,0,0
14,15x15,15,MS,False,1439,123857,NOT_EMBEDDABLE,0,0
15,15x15,15,MS,True,2774,136427,NOT_EMBEDDABLE,0,0
0,4x4,4,SS,False,44,246,OK,105,4
1,4x4,4,SS,True,60,278,OK,121,5


In [9]:
succeeded = qa_results[qa_results["status"] == "OK"]
if succeeded.empty:
    print("QA를 실행한 instance가 없습니다. 상태:", list(qa_results["status"].unique()))
else:
    columns = [
        column
        for column in (
            "instance",
            "formulation",
            "best_energy",
            "best_objective",
            "best_is_feasible",
            "feasible_fraction",
            "qa_chain_strength",
            "qa_chain_break_fraction",
            "qa_qpu_access_time_us",
            "qa_wall_clock",
        )
        if column in succeeded.columns
    ]
    display(succeeded[columns])

,instance,formulation,best_energy,best_objective,best_is_feasible,feasible_fraction,qa_chain_strength,qa_chain_break_fraction,qa_qpu_access_time_us,qa_wall_clock
0,4x4,SS,4.516780e+04,3747.6264,False,0.003,1.149410e+07,0.018977,218138.76,1.239509
1,4x4,SS,2.389643e+05,2277.6464,False,0.003,1.149854e+07,0.004517,175516.36,1.215166
2,4x4,MS,7.611137e+05,3716.2390,False,0.000,9.940841e+06,0.000508,234596.76,1.595993
3,4x4,MS,5.329211e+06,3760.8429,False,0.000,9.940841e+06,0.002354,257379.96,1.607709
4,6x6,SS,3.054411e+06,6281.4159,False,0.000,3.356186e+07,0.009899,142398.76,1.189113
5,6x6,SS,2.976467e+06,6161.2962,False,0.000,3.357158e+07,0.029626,161919.16,1.221816
8,8x8,SS,3.773204e+07,6583.5495,False,0.000,1.263163e+08,0.000530,195956.36,1.343865
9,8x8,SS,3.982366e+07,7349.4479,False,0.000,1.263163e+08,0.000503,213739.56,1.356410
12,15x15,SS,1.350276e+09,14123.1980,False,0.000,2.811347e+08,0.185085,289658.76,2.011222
